# Explainable-AI 

## Introduction
This project demonstrates three popular Class Activation Mapping (CAM) techniques for understanding deep learning model decisions in image classification. Using the Intel Image Classification dataset and a ResNet50 model, we compare **Grad-CAM**, **Grad-CAM++**, and **Score-CAM** methods to visualize which image regions influence model predictions.

**Key Features:**
- Transfer learning with ResNet50 on 6 natural scene categories
- Side-by-side comparison of CAM visualization methods
- Memory-optimized implementations for practical use
- Interactive heatmap generation and analysis

This implementation serves as both a learning resource and practical toolkit for building interpretable computer vision models.


To enhance the functionality of the CoreAI  environment, we need to install some libraries not pre-installed but required for this notebook. 

## Pre-requisites
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment:

```bash
export PROJECT_NAME="Explainable-AI"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}-myvenv --display-name="Python (${PROJECT_NAME}-myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}-myvenv)"
```

## Install Required Libraries:

Before running the following command in jupyter notebook, make sure you are in the directory where the Jupyter Notebook and virtual environment is located. This ensures the ./ path is always current. You can use the cd command to change to your project directory and pwd to verify your current directory.


In [ ]:
import os
def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")
# set_env_with_cache_dir("HF_HOME", "huggingface")
set_env_with_cache_dir('KAGGLEHUB_CACHE', 'data')


In [ ]:
!. ./myvenv/bin/activate; pip install -r requirements.txt

## Checking GPU Availability
Runs `nvidia-smi` to display GPU details, confirming hardware availability for faster training. A compatible GPU (like NVIDIA RTX A6000 shown) is recommended for optimal performance.


In [ ]:
!nvidia-smi

## Download dataset (if not already downloaded)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

## Importing Libraries
Imports all necessary libraries for deep learning, computer vision, and explainable AI functionality.

In [ ]:
import os
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
import cv2
import keras
from keras import backend as K

from keras.models import Model, Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Flatten, Input
from keras.layers import Conv2D, Activation, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input, ResNet50
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

## Global Configuration and Constants
Defines project-wide constants including image dimensions, class mappings, and training parameters.

In [ ]:
SEED = 42
DIR_PATH = os.path.join(os.environ['KAGGLEHUB_CACHE'], "datasets/puneet6060/intel-image-classification/versions/2/seg_train/seg_train")
if not os.path.exists(DIR_PATH):
    print(f"!!!!! Path ({DIR_PATH} not found -- Must fix")
W = 112 # The default size for ResNet is 224 but resize to .5 to save memory size
H = 112 # The default size for ResNet is 224 but resize to .5 to save memory size
LABEL_TO_CLASS = {
    "buildings": 0,
    "forest":    1,
    "glacier":   2,
    "mountain":  3,
    "sea":       4,
    "street":    5,
}
N_EPOCHS = 5
BATCH_SIZE = 32

CLASS_TO_LABEL = {v: k for k, v in LABEL_TO_CLASS.items()}
N_CLASSES = len(LABEL_TO_CLASS)

## Data Loading and Preprocessing Functions
Defines utility functions for loading images, splitting datasets, and building the ResNet50-based model architecture.

In [ ]:
def get_images(
        dir_path, 
        label_to_class, 
        w, 
        h, 
        seed,
    ):
    """Read images / labels from directory.
    
    Args:
        dir_path (str): Dir path saved data.
        label_to_class (dict[str, int]): Dict of label to class.
        w (int): Width size of image.
        h (int): Height size of image.
        seed (int): Random seed.
    
    Returns:
        tuple[np.ndarray, np.ndarray]: images, classes.
    """
    images = []
    classes = []
    
    for label_name in os.listdir(dir_path):
        cls = label_to_class[label_name]
        for img_name in os.listdir("/".join([dir_path, label_name])):
            img = load_img("/".join([dir_path, label_name, img_name]), target_size=(w, h))
            img = img_to_array(img)
            images.append(img)
            classes.append(cls)
            
    images = np.array(images, dtype=np.float32)
    classes = np.array(classes, dtype=np.float32)
    images, classes = shuffle(images, classes, random_state=seed)
    
    return images, classes


def split_dataset(
        images, 
        classes, 
        train_size=0.8, 
        test_size=0.2, 
        shuffle=False,
    ):
    """Split dataset.
    
    Args:
        images (np.ndarray): images.
        classes (np.ndarray): classes.
        train_size (float): Train data rate of split data.
        test_size (float): Test data rate of split data.
        shuffle (bool): Shuffle or not.
    
    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: X of train, y of train, X of test, y of test.
    """
    indices_train, indices_test = train_test_split(
        list(range(images.shape[0])), 
        train_size=train_size,
        test_size=test_size,
        shuffle=shuffle,
    )

    x_train = images[indices_train]
    y_train = classes[indices_train]
    x_test = images[indices_test]
    y_test = classes[indices_test]
    
    return x_train, y_train, x_test, y_test


def build_model(w, h, n_classes):
    """Build model function.
    
    Args:
        w (int): Width size of image.
        h (int): Height size of image.
        n_classes (int): The number of class.
        
    Returns:
        keras.engine.training.Model: Model.
    """ 
    # Resnet
    input_tensor = Input(shape=(w, h, 3)) # To change input shape
    resnet50 = ResNet50(
        include_top=False,                # To change output shape
        weights="imagenet",               # Use pre-trained model
        input_tensor=input_tensor,        # Change input shape for this task
    )
    
    # fc layer
    top_model = Sequential()
    top_model.add(GlobalAveragePooling2D())               # Add GAP for cam
    top_model.add(Dense(n_classes, activation="softmax")) # Change output shape for this task
    
    # model
    model = Model(inputs=resnet50.input, outputs=top_model(resnet50.output))
    
    # frozen weights
    for layer in model.layers[:-10]:
        layer.trainable = False or isinstance(layer, BatchNormalization) # If Batch Normalization layer, it should be trainable
        
    # compile
    model.compile(
        optimizer="adam", 
        loss="categorical_crossentropy", 
        metrics=["accuracy"],
    )
    
    return model

## Visualization and Analysis Functions
Implements functions for confusion matrix plotting, image superimposition, and CAM visualization methods.

In [ ]:

def plot_confusion_matrix(model, x_test, label_to_class):
    """Plot confusion matrix.
    
    Args:
        model (keras.engine.training.Model): Model.
        x_test (np.ndarray): X of test.
        label_to_class (dict[str, int]): Dict of label to class.
    """
    x = preprocess_input(copy.deepcopy(x_test))
    y_preds = model.predict(x)
    y_preds = np.argmax(y_preds, axis=1)
    y_trues = np.argmax(y_test, axis=1)
    cm = confusion_matrix(y_trues, y_preds)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar_kws={"shrink": .3}, linewidths=.1, ax=ax)
    ax.set(
        xticklabels=list(label_to_class.keys()),
        yticklabels=list(label_to_class.keys()),
        title="confusion matrix",
        ylabel="True label",
        xlabel="Predicted label",
    )
    params = dict(rotation=45, ha="center", rotation_mode="anchor")
    plt.setp(ax.get_yticklabels(), **params)
    plt.setp(ax.get_xticklabels(), **params)
    plt.show()
    
    
def superimpose(img, cam):
    """Superimpose original image and cam heatmap.
    
    Args:
        img (np.ndarray): Image.
        cam (np.ndarray): Cam image.
        
    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: Image, heatmap, superimposed image.
    """
    heatmap = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * .5 + img * .5
    superimposed_img = np.minimum(superimposed_img, 255.0).astype(np.uint8) # scale 0 to 255  
    superimposed_img = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)
    
    return img, heatmap, superimposed_img


def plot_org_img_and_cam_heatmap(
        model, 
        cam_func, 
        superimpose, 
        img, 
        cls_true, 
        class_to_label, 
        cam_name,
    ):
    """Plot original image, heatmap from cam and superimpose image.
    
    Args:
        model (keras.engine.training.Model): Model.
        cam_func (function): Cam function.
        superimpose (function): Superimpose function.
        img (np.ndarray): Image.
        cls_true (float): Class of given img.
        class_to_label (dict[int, str]): Dict of class to label.
        cam_name (str): Used cam name.
    """
    # for cam
    x = np.expand_dims(img, axis=0)
    x = preprocess_input(copy.deepcopy(x))

    # for superimpose
    img = np.uint8(img)

    # cam / superimpose
    cls_pred, cam = cam_func(model=model, x=x, layer_name=model.layers[-2].name)
    img, heatmap, superimposed_img = superimpose(img, cam)

    fig, axs = plt.subplots(ncols=3, figsize=(9, 4))

    axs[0].imshow(img)
    axs[0].set_title("original image")
    axs[0].axis("off")

    axs[1].imshow(heatmap)
    axs[1].set_title("heatmap")
    axs[1].axis("off")

    axs[2].imshow(superimposed_img)
    axs[2].set_title("superimposed image")
    axs[2].axis("off")

    title = "CAM name: " + cam_name + " / True label: " + class_to_label[cls_true] + " / Predicted label : " + class_to_label[cls_pred]
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    



## Grad-CAM (Gradient-weighted Class Activation Mapping) Implementation
Implements Grad-CAM algorithm for generating class activation maps using gradient-based attention.

In [ ]:
def grad_cam(model, x, layer_name):
    # Ensure input is a tensor and has batch dimension
    if not isinstance(x, tf.Tensor):
        x = tf.convert_to_tensor(x)
    if len(x.shape) == 3:
        x = tf.expand_dims(x, axis=0)

    # Get the target layer
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(layer_name).output, model.output]
    )

    # Record operations for automatic differentiation
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(x)
        cls = tf.argmax(predictions[0])
        loss = predictions[:, cls]

    # Compute gradients of the output neuron (target class) with respect to the feature map
    grads = tape.gradient(loss, conv_outputs)

    # Compute guided gradients and weights
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    # Weight the channels by corresponding gradients
    cam = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    # Apply ReLU and normalize
    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = cam.numpy()

    return int(cls.numpy()), cam


## Grad-CAM++ Implementation
Implements advanced Grad-CAM++ method with improved localization using higher-order gradients.

In [ ]:
def grad_cam_plus_plus(model, x, layer_name):
    """
    Grad-CAM++ function using eager execution.

    Args:
        model (tf.keras.Model): The trained model.
        x (np.ndarray or tf.Tensor): Input image, shape (H, W, 3) or (1, H, W, 3).
        layer_name (str): Name of the target convolutional layer.

    Returns:
        tuple[int, np.ndarray]: (Predicted class index, Grad-CAM++ heatmap)
    """
    # Ensure input is a tensor and has batch dimension
    if not isinstance(x, tf.Tensor):
        x = tf.convert_to_tensor(x)
    if len(x.shape) == 3:
        x = tf.expand_dims(x, axis=0)

    # Build a model that outputs both the target layer and predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape1:
        with tf.GradientTape() as tape2:
            with tf.GradientTape() as tape3:
                conv_outputs, predictions = grad_model(x)
                cls = tf.argmax(predictions[0])
                loss = predictions[:, cls]

            # First derivative
            grads = tape3.gradient(loss, conv_outputs)
        # Second derivative
        first_derivative = grads
        second_derivative = tape2.gradient(first_derivative, conv_outputs)
    # Third derivative
    third_derivative = tape1.gradient(second_derivative, conv_outputs)

    # Compute alpha coefficients
    global_sum = tf.reduce_sum(conv_outputs, axis=[1, 2], keepdims=True)
    alpha_num = second_derivative
    alpha_denom = 2.0 * second_derivative + third_derivative * global_sum
    alpha_denom = tf.where(alpha_denom != 0.0, alpha_denom, tf.ones_like(alpha_denom))
    alphas = alpha_num / alpha_denom

    # Normalize alphas
    alphas = tf.where(tf.math.is_nan(alphas), tf.zeros_like(alphas), alphas)
    weights = tf.reduce_sum(tf.nn.relu(first_derivative) * alphas, axis=[1, 2])

    # Compute the Grad-CAM++ heatmap
    conv_outputs = conv_outputs[0]
    weights = weights[0]
    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)

    # Apply ReLU and normalize
    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = cam.numpy()

    return int(cls.numpy()), cam


def softmax(x):
    """Softmax function.
    
    Args:
        x (np.ndarray): Input.
        
    Returns:
        np.ndarray: Softmax(x)
    """
    return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)

## Score-CAM Implementation
Implements Score-CAM method that uses forward passing scores instead of gradients for explanation generation.


In [ ]:
def score_cam(model, x, layer_name, max_N=64, batch_size=16):
    """
    Memory-efficient Score-CAM function using eager execution with batch processing.

    Args:
        model (tf.keras.Model): The trained model.
        x (np.ndarray or tf.Tensor): Input image, shape (H, W, 3) or (1, H, W, 3).
        layer_name (str): Name of the target convolutional layer.
        max_N (int): Maximum number of activation maps to use. Default 64 to avoid OOM.
        batch_size (int): Batch size for processing masked inputs. Default 16.

    Returns:
        tuple[int, np.ndarray]: (Predicted class index, Score-CAM heatmap)
    """
    # Ensure input is a tensor and has batch dimension
    if not isinstance(x, tf.Tensor):
        x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 3:
        x = tf.expand_dims(x, axis=0)

    # Predict class index
    preds = model(x)
    cls = tf.argmax(preds[0]).numpy()

    # Get activation maps from the target layer
    activation_model = tf.keras.Model(model.inputs, model.get_layer(layer_name).output)
    act_map_array = activation_model(x).numpy()  # shape: (1, H', W', C)

    # Select top-N most variant activation maps to reduce memory usage
    if max_N != -1 and act_map_array.shape[3] > max_N:
        act_map_std_list = [np.std(act_map_array[0, :, :, k]) for k in range(act_map_array.shape[3])]
        unsorted_max_indices = np.argpartition(-np.array(act_map_std_list), max_N)[:max_N]
        max_N_indices = unsorted_max_indices[np.argsort(-np.array(act_map_std_list)[unsorted_max_indices])]
        act_map_array = act_map_array[:, :, :, max_N_indices]

    # Resize activation maps to input image size
    input_shape = x.shape[1:3]
    act_map_resized_list = [
        cv2.resize(act_map_array[0, :, :, k], (input_shape[1], input_shape[0]), interpolation=cv2.INTER_LINEAR)
        for k in range(act_map_array.shape[3])
    ]

    # Normalize activation maps to [0, 1]
    act_map_normalized_list = []
    for act_map_resized in act_map_resized_list:
        min_val = np.min(act_map_resized)
        max_val = np.max(act_map_resized)
        if max_val - min_val != 0:
            act_map_normalized = (act_map_resized - min_val) / (max_val - min_val)
        else:
            act_map_normalized = act_map_resized
        act_map_normalized_list.append(act_map_normalized)

    # Create masked inputs
    masked_input_list = []
    x_np = x.numpy() if isinstance(x, tf.Tensor) else x
    for act_map_normalized in act_map_normalized_list:
        masked_input = np.copy(x_np)
        for k in range(3):
            masked_input[0, :, :, k] *= act_map_normalized
        masked_input_list.append(masked_input)
    masked_input_array = np.concatenate(masked_input_list, axis=0)

    # Process masked inputs in batches to avoid OOM
    pred_from_masked_input_list = []
    for i in range(0, len(masked_input_array), batch_size):
        batch = masked_input_array[i:i+batch_size]
        batch_preds = model(batch)
        batch_preds = tf.nn.softmax(batch_preds, axis=-1).numpy()
        pred_from_masked_input_list.append(batch_preds)
    
    pred_from_masked_input_array = np.concatenate(pred_from_masked_input_list, axis=0)

    # Use the score for the target class as weights
    weights = pred_from_masked_input_array[:, cls]

    # Weighted combination of activation maps
    cam = np.dot(np.stack(act_map_resized_list, axis=-1), weights)
    cam = np.maximum(cam, 0)
    cam = cam / (np.max(cam) + 1e-8)

    return int(cls), cam




## CAM Comparison Function
Provides comprehensive comparison visualization across all three CAM methods for targeted class analysis.

In [ ]:
def compare_each_cam(
        images, 
        classes, 
        class_to_label, 
        model, 
        grad_cam, 
        grad_cam_plus_plus, 
        score_cam, 
        superimpose, 
        layer_name, 
        target_cls,
    ):
    """Compare Grad-CAM / Grad-CAM++ / Score-CAM on target class images.
    
    Args:
        images (np.ndarray): images.
        classes (np.ndarray): classes.
        class_to_label (dict[int, str]): Dict of class to label.
        model (keras.engine.training.Model): Model.
        grad_cam (function): Grad-CAM function.
        grad_cam_plus_plus (function): Grad-CAM++ function.
        score_cam (function): Score-CAM function.
        superimpose (function): Superimpose function.
        layer_name (str): Get layer name.
        target_cls (int): Target class.
    """
    indices = np.where(classes == target_cls)[0]
    label = class_to_label[target_cls]

    n_cols = 10 # # of sample plot

    fig, axs = plt.subplots(ncols=n_cols, nrows=4, figsize=(25, 9))

    for i in range(n_cols):
        
        img = images[indices[i]]
        
        # for cam
        x = np.expand_dims(img, axis=0)
        x = preprocess_input(copy.deepcopy(x))

        # original
        axs[0, i].imshow(np.uint8(img))
        axs[0, i].set_title(label)
        axs[0, i].set_xticks([])
        axs[0, i].set_yticks([])
        if i == 0:
            axs[0, i].set_ylabel("Original", rotation=0, ha="right")

        # Grad-CAM
        cls_pred, cam = grad_cam(model=model, x=x, layer_name=layer_name)
        _, _, img_grad_cam = superimpose(img, cam)
        axs[1, i].imshow(img_grad_cam)
        axs[1, i].set_title("pred: " + class_to_label[cls_pred])
        axs[1, i].set_xticks([])
        axs[1, i].set_yticks([])
        if i == 0:
            axs[1, i].set_ylabel("Grad-CAM", rotation=0, ha="right")

        # Grad-CAM++
        cls_pred, cam = grad_cam_plus_plus(model=model, x=x, layer_name=layer_name)
        _, _, img_grad_cam_plus_plus = superimpose(img, cam)
        axs[2, i].imshow(img_grad_cam_plus_plus)
        axs[2, i].set_title("pred: " + class_to_label[cls_pred])
        axs[2, i].set_xticks([])
        axs[2, i].set_yticks([])
        if i == 0:
            axs[2, i].set_ylabel("Grad-CAM++", rotation=0, ha="right")

        # Score-CAM
        cls_pred, cam = score_cam(model=model, x=x, layer_name=layer_name)
        _, _, img_score_cam = superimpose(img, cam)
        axs[3, i].imshow(img_score_cam)
        axs[3, i].set_title("pred: " + class_to_label[cls_pred])
        axs[3, i].set_xticks([])
        axs[3, i].set_yticks([])
        if i == 0:
            axs[3, i].set_ylabel("Score-CAM", rotation=0, ha="right")

    plt.show()

## Dataset Loading and Exploration
Loads the complete dataset and displays shape information for images and class labels.



In [ ]:
images, classes = get_images(
    dir_path=DIR_PATH, 
    label_to_class=LABEL_TO_CLASS,
    w=W,
    h=H,
    seed=SEED,
)

images.shape, classes.shape

## Class Distribution Analysis
Analyzes and displays the distribution of images across different classes to understand dataset balance.

In [ ]:
# Check the number of images for each labels
n_total_images = images.shape[0]
for target_cls in list(CLASS_TO_LABEL.keys()):
    indices = np.where(classes == target_cls)[0] # get target class indices on images / classes
    n_target_cls = indices.shape[0]
    label = CLASS_TO_LABEL[target_cls]
    print(label, ":", n_target_cls, round(n_target_cls / n_total_images, 2))

## Sample Image Visualization
Displays sample images from each class to visually inspect the dataset quality and variety.

In [ ]:
# Visualize some images / labels

for target_cls in list(CLASS_TO_LABEL.keys()):
    indices = np.where(classes == target_cls)[0] # get target class indices on images / classes
    label = CLASS_TO_LABEL[target_cls]

    n_cols = 10 # The number of sample plot
    fig, axs = plt.subplots(ncols=n_cols, figsize=(25, 3))

    for i in range(n_cols):
        axs[i].imshow(np.uint8(images[indices[i]]))
        axs[i].axis('off')
        axs[i].set_title(label)

    plt.show()

## Data Splitting and Augmentation
Splits dataset into train/test sets and configures data augmentation generators for improved model training.

In [ ]:
# split dataset to train and test

x_train, y_train, x_test, y_test = split_dataset(images, classes)

# to one-hot

y_train = keras.utils.to_categorical(y_train, N_CLASSES)
y_test = keras.utils.to_categorical(y_test, N_CLASSES)

# To image data generator
# Use image data augmentation for train data

datagen_train = ImageDataGenerator(
    preprocessing_function=preprocess_input, # image preprocessing function
    rotation_range=30,                       # randomly rotate images in the range
    zoom_range=0.1,                          # Randomly zoom image
    width_shift_range=0.1,                   # randomly shift images horizontally
    height_shift_range=0.1,                  # randomly shift images vertically
    horizontal_flip=True,                    # randomly flip images horizontally
    vertical_flip=False,                     # randomly flip images vertically
)
datagen_test = ImageDataGenerator(
    preprocessing_function=preprocess_input, # image preprocessing function
)

## Model Architecture Setup
Builds and displays the ResNet50-based model architecture with custom classification head.

In [ ]:
model = build_model(w=W, h=H, n_classes=N_CLASSES)
model.summary()

## Model Training
Trains the model using augmented data with validation monitoring over specified epochs.

In [ ]:
history = model.fit(
    datagen_train.flow(
        x_train, 
        y_train, 
        batch_size=BATCH_SIZE,
    ),
    epochs=N_EPOCHS,
    validation_data=datagen_test.flow(
        x_test, 
        y_test, 
        batch_size=BATCH_SIZE,
    ),
)

## Model Performance Evaluation
 Evaluates trained model performance by generating and visualizing confusion matrix on test data.

In [ ]:
# Confirm the result of finetuning
# Plot the confusion matrix
plot_confusion_matrix(model, x_test, LABEL_TO_CLASS)

## Individual CAM Method Demonstration
Demonstrates each CAM method individually on a sample image with memory management between methods.

In [ ]:
import gc
import tensorflow as tf

# Run Grad-CAM
plot_org_img_and_cam_heatmap(
    model=model, 
    cam_func=grad_cam,
    superimpose=superimpose,
    img=images[0],
    cls_true=classes[0], 
    class_to_label=CLASS_TO_LABEL, 
    cam_name="Grad-CAM",
)

# Clear memory before next method
tf.keras.backend.clear_session()
gc.collect()

# Run Grad-CAM++
plot_org_img_and_cam_heatmap(
    model=model, 
    cam_func=grad_cam_plus_plus,
    superimpose=superimpose,
    img=images[0],
    cls_true=classes[0], 
    class_to_label=CLASS_TO_LABEL, 
    cam_name="Grad-CAM++",
)

# Clear memory before Score-CAM
tf.keras.backend.clear_session()
gc.collect()

# Run Score-CAM with minimal settings
plot_org_img_and_cam_heatmap(
    model=model, 
    cam_func=score_cam,
    superimpose=superimpose,
    img=images[0],
    cls_true=classes[0], 
    class_to_label=CLASS_TO_LABEL, 
    cam_name="Score-CAM",
)

## Comprehensive CAM Comparison
Runs comparative analysis across all CAM methods for each class to evaluate explanation quality differences.

In [ ]:
for k, v in LABEL_TO_CLASS.items():
    compare_each_cam(
        images=images, 
        classes=classes, 
        class_to_label=CLASS_TO_LABEL,
        model=model,
        grad_cam=grad_cam, 
        grad_cam_plus_plus=grad_cam_plus_plus, 
        score_cam=score_cam, 
        superimpose=superimpose, 
        layer_name=model.layers[-2].name, 
        target_cls=v,
    )